In [ ]:
# Imports and setup
import fsspec
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.compute as pc

import os
import time
from datetime import datetime

print("All libraries imported successfully!")

All libraries imported successfully!


## Remote Schema expolaration

In [2]:
def get_column_metadata(URL, column_names=None, detailed=False):
    
    with fsspec.open(URL, "rb") as f:
        parquet_file = pq.ParquetFile(f)
        metadata = parquet_file.metadata
        schema = parquet_file.schema_arrow
        
        # Handle column selection
        if column_names is None:
            column_names = schema.names
        elif isinstance(column_names, str):
            column_names = [column_names]
        
        # Validate columns exist
        valid_columns = []
        for col_name in column_names:
            if col_name in schema.names:
                valid_columns.append(col_name)
            else:
                print(f"Warning: Column '{col_name}' not found in schema. Available columns: {schema.names}")
        
        if not valid_columns:
            print("No valid columns to analyze")
            return
        
        mode = "DETAILED" if detailed else "OVERVIEW"
        print(f"=== {mode} FOR {len(valid_columns)} COLUMN(S) ===")
        
        # Process each column
        for column_name in valid_columns:
            col_idx = schema.names.index(column_name)
            col_type = schema.field(column_name).type
            
            if detailed:
                print(f"\n{'='*50}")
                print(f"DETAILED METADATA FOR '{column_name}'")
                print(f"{'='*50}")
            else:
                print(f"\n{column_name}: {col_type}")
            
            # Initialize aggregators
            total_nulls = 0
            total_values = 0
            has_min_max = False
            global_min = None
            global_max = None
            has_distinct_count = False
            distinct_count = None
            
            # Aggregate across all row groups
            for rg_idx in range(metadata.num_row_groups):
                rg_metadata = metadata.row_group(rg_idx)
                col_metadata = rg_metadata.column(col_idx)
                rg_num_rows = rg_metadata.num_rows
                
                if col_metadata.statistics:
                    stats = col_metadata.statistics
                    
                    # Aggregate null count
                    if stats.has_null_count:
                        total_nulls += stats.null_count
                    
                    # Aggregate min/max values
                    if stats.has_min_max:
                        has_min_max = True
                        try:
                            current_min = stats.min
                            current_max = stats.max
                            
                            # Handle different data types for comparison
                            if pa.types.is_timestamp(col_type):
                                current_min = pa.scalar(current_min).cast(col_type).as_py()
                                current_max = pa.scalar(current_max).cast(col_type).as_py()
                            
                            # Update global min/max
                            if global_min is None or current_min < global_min:
                                global_min = current_min
                            if global_max is None or current_max > global_max:
                                global_max = current_max
                                
                        except (OverflowError, Exception):
                            # If we can't compare, just take the first available values
                            if global_min is None:
                                global_min = stats.min
                                global_max = stats.max
                    
                    # Handle distinct count (take max across row groups as approximation)
                    if stats.has_distinct_count:
                        has_distinct_count = True
                        if distinct_count is None or stats.distinct_count > distinct_count:
                            distinct_count = stats.distinct_count
                
                total_values += rg_num_rows
            
            # Calculate percentages
            null_percentage = (total_nulls / total_values) * 100 if total_values > 0 else 0
            non_null_count = total_values - total_nulls
            
            # Display results based on detail level
            if detailed:
                print(f"Data type: {col_type}")
                print(f"Total rows: {total_values:,}")
                print(f"Null count: {total_nulls:,} ({null_percentage:.2f}%)")
                print(f"Non-null count: {non_null_count:,}")
                
                if has_min_max:
                    print(f"\nValue Range:")
                    try:
                        if pa.types.is_timestamp(col_type):
                            print(f"  Min: {global_min}")
                            print(f"  Max: {global_max}")
                        elif pa.types.is_string(col_type) or pa.types.is_large_string(col_type):
                            min_str = str(global_min)
                            max_str = str(global_max)
                            if len(min_str) > 50:
                                min_str = min_str[:47] + "..."
                            if len(max_str) > 50:
                                max_str = max_str[:47] + "..."
                            print(f"  Min: {min_str}")
                            print(f"  Max: {max_str}")
                        else:
                            print(f"  Min: {global_min}")
                            print(f"  Max: {global_max}")
                    except Exception as e:
                        print(f"  Min: <error displaying: {e}>")
                        print(f"  Max: <error displaying: {e}>")
                else:
                    print(f"\nValue Range: No min/max statistics available")
                
                if has_distinct_count:
                    print(f"Distinct values: {distinct_count:,}")
                else:
                    print(f"Distinct values: No distinct count statistics available")
                    
            else:
                # Quick overview mode
                print(f"  Total rows: {total_values:,}")
                print(f"  Null count: {total_nulls:,} ({null_percentage:.1f}%)")
                print(f"  Non-null: {non_null_count:,}")
                
                if has_min_max:
                    try:
                        if pa.types.is_timestamp(col_type):
                            print(f"  Value range: {global_min} to {global_max}")
                        elif pa.types.is_string(col_type) or pa.types.is_large_string(col_type):
                            min_str = str(global_min)[:20] + "..." if len(str(global_min)) > 20 else str(global_min)
                            max_str = str(global_max)[:20] + "..." if len(str(global_max)) > 20 else str(global_max)
                            print(f"  Value range: '{min_str}' to '{max_str}'")
                        else:
                            print(f"  Value range: {global_min} to {global_max}")
                    except Exception:
                        print(f"  Value range: Available (display error)")
                else:
                    print(f"  Value range: No statistics available")

## Posts

In [3]:
POSTS_URL = "https://bsky-data.leobalduf.com/posts.parquet"

# First, just understand the structure
with fsspec.open(POSTS_URL, "rb") as f:
    parquet_file = pq.ParquetFile(f)
    print("Posts Database")
    print("Columns:", parquet_file.schema.names)
    print("Number of rows:", parquet_file.metadata.num_rows)
    print("Number of row groups:", parquet_file.metadata.num_row_groups)

Posts Database
Columns: ['did_id', 'rkey', 'created_at', 'languages', 'labels', 'tags', 'embed_type', 'did_id', 'collection', 'rkey', 'embed_external_uri', 'embed_images', 'embed_media', 'embed_video', 'did_id', 'collection', 'rkey', 'did_id', 'collection', 'rkey']
Number of rows: 1290686413
Number of row groups: 10495


In [4]:
get_column_metadata(POSTS_URL, ["did_id", "created_at"], detailed=True)

=== DETAILED FOR 2 COLUMN(S) ===

DETAILED METADATA FOR 'did_id'
Data type: int64
Total rows: 1,290,686,413
Null count: 0 (0.00%)
Non-null count: 1,290,686,413

Value Range:
  Min: 3
  Max: 34251847
Distinct values: No distinct count statistics available

DETAILED METADATA FOR 'created_at'
Data type: timestamp[us, tz=UTC]
Total rows: 1,290,686,413
Null count: 76 (0.00%)
Non-null count: 1,290,686,337

Value Range:
  Min: 0001-01-01 00:00:00+00:00
  Max: 9999-12-31 23:59:59.999000+00:00
Distinct values: No distinct count statistics available


## Profiles

In [5]:

PROFILES_URL = "https://bsky-data.leobalduf.com/profiles.parquet"

# First, just understand the structure
with fsspec.open(PROFILES_URL, "rb") as f:
    parquet_file = pq.ParquetFile(f)
    print("Profiles Database")
    print("Columns:", parquet_file.schema.names)
    print("Number of rows:", parquet_file.metadata.num_rows)
    print("Number of row groups:", parquet_file.metadata.num_row_groups)

Profiles Database
Columns: ['did_id', 'rkey', 'created_at', 'description', 'labels', '/', 'size', 'mimeType', '/', 'size', 'mimeType', 'joined_via_starter_pack', 'additional_fields']
Number of rows: 32170299
Number of row groups: 262


In [6]:
get_column_metadata(PROFILES_URL, ["did_id", "created_at", "joined_via_starter_pack"], detailed=True)

=== DETAILED FOR 3 COLUMN(S) ===

DETAILED METADATA FOR 'did_id'
Data type: int64
Total rows: 32,170,299
Null count: 0 (0.00%)
Non-null count: 32,170,299

Value Range:
  Min: 1
  Max: 34251851
Distinct values: No distinct count statistics available

DETAILED METADATA FOR 'created_at'
Data type: timestamp[us, tz=UTC]
Total rows: 32,170,299
Null count: 4,247,596 (13.20%)
Non-null count: 27,922,703

Value Range:
  Min: 0081-11-15 02:48:08.855000+00:00
  Max: 2954-08-31 07:36:16.393000+00:00
Distinct values: No distinct count statistics available

DETAILED METADATA FOR 'joined_via_starter_pack'
Data type: extension<arrow.json>
Total rows: 32,170,299
Null count: 1,488,624 (4.63%)
Non-null count: 30,681,675

Value Range:
  Min: application/octet-stream
  Max: text/html
Distinct values: 5


# Cleaning
In both posts and profiles I'm just interested in a few columns:
1) Posts: ["did_id", "created_at"]
2) Profiles: ["did_id", "created_at", "joined_via_starter_pack"]

In [7]:
# Define cleaning rules per database/table
CLEANING_RULES = {
    'default': {
        'created_at': {
            'remove_null': True,
            'max_date': '2025-05-14',
            'min_date': '2020-01-01'   # Filter out very old dates
        },
        'did_id': {
            'remove_null': True,
            'remove_empty_strings': True
        },
        'joined_via_starter_pack': {
            'fill_null': False
        }
    },
    'posts': {
        'did_id': {
            'remove_null': True,
            'remove_empty_strings': True
        },
        'created_at': {
            'remove_null': True,
            'max_date': '2025-05-14', 
            'min_date': '2020-01-01'   # Filter out very old dates
        },
    },
    'profiles': {
        'did_id': {
            'remove_null': True,
            'remove_empty_strings': True
        },
        'created_at': {
            'remove_null': True,
            'max_date': '2025-05-14', 
            'min_date': '2020-01-01'   # Filter out very old dates
        },
    }
}

In [8]:
def clean_table_with_rules(table, rules, table_type='default'):
    """
    Clean table with memory-efficient filtering for large Parquet files.
    Applies filters incrementally to avoid large intermediate masks.
    """
    
    current_table = table
    applied_rules = []
    rows_removed_log = []
    
    rule_set = rules.get(table_type, {})
    
    # Process each column's rules sequentially
    for column_name, column_rules in rule_set.items():
        if column_name not in current_table.column_names:
            continue
            
        original_rows = current_table.num_rows
        array = current_table[column_name]
        
        # Build individual mask for this column's rules
        column_mask = None
        
        # Rule: remove nulls
        if column_rules.get('remove_null'):
            null_mask = pc.is_valid(array)
            column_mask = null_mask if column_mask is None else pc.and_(column_mask, null_mask)
            applied_rules.append(f"removed_null_{column_name}")
        
        # Rule: remove empty strings
        if column_rules.get('remove_empty_strings') and pa.types.is_string(array.type):
            non_empty_mask = pc.not_equal(array, "")
            column_mask = non_empty_mask if column_mask is None else pc.and_(column_mask, non_empty_mask)
            applied_rules.append(f"removed_empty_{column_name}")
        
        # Rule: date range filtering
        if column_rules.get('max_date') and pa.types.is_timestamp(array.type):
            max_date = datetime.strptime(column_rules['max_date'], '%Y-%m-%d')
            max_date_scalar = pa.scalar(max_date, type=array.type)
            max_date_mask = pc.less_equal(array, max_date_scalar)
            column_mask = max_date_mask if column_mask is None else pc.and_(column_mask, max_date_mask)
            applied_rules.append(f"max_date_{column_name}")
            
        if column_rules.get('min_date') and pa.types.is_timestamp(array.type):
            min_date = datetime.strptime(column_rules['min_date'], '%Y-%m-%d')
            min_date_scalar = pa.scalar(min_date, type=array.type)
            min_date_mask = pc.greater_equal(array, min_date_scalar)
            column_mask = min_date_mask if column_mask is None else pc.and_(column_mask, min_date_mask)
            applied_rules.append(f"min_date_{column_name}")
        
        # Rule: string length
        if column_rules.get('min_length') and pa.types.is_string(array.type):
            length_mask = pc.greater(pc.utf8_length(array), column_rules['min_length'])
            column_mask = length_mask if column_mask is None else pc.and_(column_mask, length_mask)
            applied_rules.append(f"min_length_{column_name}")
        
        # Apply this column's filter and immediately clean up
        if column_mask is not None:
            current_table = current_table.filter(column_mask)
            rows_removed = original_rows - current_table.num_rows
            if rows_removed > 0:
                rows_removed_log.append(f"{column_name}: {rows_removed:,} rows")
    
    return current_table, applied_rules, rows_removed_log

In [13]:
def clean_large_parquet_with_rules(input_path, output_path, rules, table_type='default', batch_size=100000):
    """Clean very large Parquet files by processing in batches, keeping only specified columns"""
    
    # Get original file size
    original_size = os.path.getsize(input_path)
    print(f"Cleaning large file ({original_size / (1024**3):.2f} GB) in batches of {batch_size:,} rows...")
    
    # Read the schema first to check available columns
    schema = pq.read_schema(input_path)
    available_columns = set(schema.names)
    
    # Determine which columns to read based on cleaning rules
    rule_columns = set(rules.get(table_type, {}).keys())
    columns_to_read = list(rule_columns.intersection(available_columns))
    
    # Check for missing columns
    missing_columns = rule_columns - available_columns
    if missing_columns:
        print(f"Warning: Columns not found in input file: {missing_columns}")
    
    if not columns_to_read:
        raise ValueError(f"No valid columns found for table type '{table_type}'. Available: {available_columns}")
    
    print(f"Processing columns: {columns_to_read}")
    
    # Set up Parquet writer
    writer = None
    total_rows_processed = 0
    total_rows_kept = 0
    
    # Process in batches - only read the columns we need
    for batch in pq.read_table(input_path, columns=columns_to_read).to_batches(max_chunksize=batch_size):
        total_rows_processed += batch.num_rows
        
        # Convert batch to table and apply cleaning rules
        batch_table = pa.Table.from_batches([batch])
        cleaned_batch, applied_rules, removal_log = clean_table_with_rules(batch_table, rules, table_type)
        
        total_rows_kept += cleaned_batch.num_rows
        
        # Initialize writer with first cleaned batch schema
        if writer is None:
            writer = pq.ParquetWriter(output_path, cleaned_batch.schema)
            print(f"Applied cleaning rules: {applied_rules}")
            if removal_log:
                print(f"Row removal by column: {removal_log}")
        
        # Write cleaned batch (only contains our target columns)
        writer.write_table(cleaned_batch)
    
    if writer:
        writer.close()
        
    # Calculate file size reduction
    final_size = os.path.getsize(output_path)
    size_reduction = original_size - final_size
    size_reduction_percent = (size_reduction / original_size * 100) if original_size > 0 else 0
    
    retention_rate = (total_rows_kept / total_rows_processed * 100) if total_rows_processed > 0 else 0

    print("\n" + "="*60)
    print("CLEANING SUMMARY:")
    print("="*60)
    print(f"Rows:      {total_rows_processed:,} → {total_rows_kept:,} ({retention_rate:.1f}% kept)")
    print(f"File size: {original_size / (1024**3):.2f} GB → {final_size / (1024**3):.2f} GB")
    print(f"Reduction: {size_reduction / (1024**3):.2f} GB ({size_reduction_percent:.1f}% smaller)")
    print(f"Columns:   {len(schema.names)} → {len(columns_to_read)}")
    print(f"Output:    {output_path}")
    print("="*60)
    

In [14]:
input_path = "/home/ale/Documents/uni/mp/data/raw/profiles.parquet"
output_path = "/home/ale/Documents/uni/mp/data/cleaned/profiles_cleaned.parquet"

clean_large_parquet_with_rules(
    input_path, 
    output_path,
    CLEANING_RULES,
    'profiles',
    batch_size= 50000
)

Cleaning large file (1.77 GB) in batches of 50,000 rows...
Processing columns: ['did_id', 'created_at']
Applied cleaning rules: ['removed_null_did_id', 'removed_null_created_at', 'max_date_created_at', 'min_date_created_at']
Row removal by column: ['created_at: 6,594 rows']

CLEANING SUMMARY:
Rows:      32,170,299 → 27,922,675 (86.8% kept)
File size: 1.77 GB → 0.41 GB
Reduction: 1.36 GB (76.8% smaller)
Columns:   9 → 2
Output:    /home/ale/Documents/uni/mp/data/cleaned/profiles_cleaned.parquet


In [ ]:
input_path = "/home/ale/Documents/uni/mp/data/raw/chunk_0_posts.parquet"
output_path = "/home/ale/Documents/uni/mp/data/cleaned/chunk_0_posts_cleaned.parquet"

clean_database(
    input_path, 
    output_path,
    target_columns=["did_id", "created_at"]
)